<h2 align="center">Machine Learning Model Development</h2>

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    BaggingRegressor,
    VotingRegressor
)
from sklearn.svm import SVR
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import GridSearchCV
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [2]:
df = pd.read_excel("../data/processed/5_year_mine_data_cleaned.xlsx")
df.columns = df.columns.str.strip()
df.head()

,YEAR,SEASON,MONTH,LOCATION,WEEK,Temperature,Relative_humidity,RF-CUMULATIVE,Wind_speed,Road_width,Distance,Production,Vehicle Capacity,PM10,PM2.5
0,2023,WINTER,DECEMBER,HAUL ROAD,1,33.0,65.0,1.0,2.55,12.5,25,157650,25000,62.4,42.6
1,2023,WINTER,DECEMBER,HAUL ROAD,1,30.1,64.5,1.0,2.55,12.5,25,157650,25000,56.8,42.6
2,2023,WINTER,DECEMBER,HAUL ROAD,1,29.2,70.3,1.0,2.55,12.5,25,157650,25000,62.4,41.9
3,2023,WINTER,DECEMBER,HAUL ROAD,2,27.6,68.0,0.0,2.55,12.5,25,157650,25000,63.9,43.6
4,2023,WINTER,DECEMBER,HAUL ROAD,2,24.5,85.3,0.0,2.55,12.5,25,157650,25000,65.7,41.9


In [3]:
features = [
        "Temperature",
        "Relative_humidity",
        "RF-CUMULATIVE",
        "Wind_speed",
        "Road_width",
        "Distance",
        "Production",
        "Vehicle Capacity",
]

target_pm10 = "PM10"
target_pm25 = "PM2.5"

In [4]:
X = df[features]
y = df[target_pm10]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

<h2 align="center">Linear Regression Model</h2>

In [5]:
from sklearn.impute import SimpleImputer

# Impute missing feature values (use median to be robust to outliers)
imputer = SimpleImputer(strategy="median")
imputer.fit(X_train)

X_train_imputed = pd.DataFrame(
	imputer.transform(X_train), columns=X_train.columns, index=X_train.index
)
X_test_imputed = pd.DataFrame(
	imputer.transform(X_test), columns=X_test.columns, index=X_test.index
)

lr = LinearRegression()
lr.fit(X_train_imputed, y_train)
pred = lr.predict(X_test_imputed)

In [6]:
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print(mae)
print(rmse)
print(r2)

9.217208450792315
11.469378856139729
0.13177627050064056


<h2 align="center">Decision Tree Model</h2>

In [7]:
# Decision Tree
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)

In [8]:
mae = mean_absolute_error(y_test, dt_pred)
rmse = np.sqrt(mean_squared_error(y_test, dt_pred))
r2 = r2_score(y_test, dt_pred)

print(mae)
print(rmse)
print(r2)

6.882729426356233
9.378589302342506
0.41946702875199127


<h2 align="center">Random Forest Model</h2>

In [9]:
# Random Forest
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

In [10]:
mae = mean_absolute_error(y_test, rf_pred)
rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
r2 = r2_score(y_test, rf_pred)

print(mae)
print(rmse)
print(r2)

6.818177827535946
9.222289779477741
0.4386556148913301


<h2 align="center">Bagging Model</h2>

In [11]:
# Bagging
try:
    bagging = BaggingRegressor(
        estimator=DecisionTreeRegressor(random_state=42),
        n_estimators=100,
        random_state=42
    )
except TypeError:
    bagging = BaggingRegressor(
        base_estimator=DecisionTreeRegressor(random_state=42),
        n_estimators=100,
        random_state=42
    )

bagging.fit(X_train, y_train)
bagging_pred = bagging.predict(X_test)

In [12]:
mae = mean_absolute_error(y_test, bagging_pred)
rmse = np.sqrt(mean_squared_error(y_test, bagging_pred))
r2 = r2_score(y_test, bagging_pred)

print(mae)
print(rmse)
print(r2)

6.831090843659066
9.222888061219537
0.4385827798366704


<h2 align="center">SVR Model</h2>

In [13]:
# SVR
svr = SVR(kernel="rbf")
svr.fit(X_train, y_train)
svr_pred = svr.predict(X_test)

In [14]:
mae = mean_absolute_error(y_test, svr_pred)
rmse = np.sqrt(mean_squared_error(y_test, svr_pred))
r2 = r2_score(y_test, svr_pred)

print(mae)
print(rmse)
print(r2)

9.916575152961991
12.391158232508358
-0.01338777290744142


<h2 align="center">XGBoost Model</h2>

In [15]:
# XGBoost
xgb_model = xgb.XGBRegressor(
    objective="reg:squarederror",
    n_estimators=200,
    learning_rate=0.1,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

In [16]:
mae = mean_absolute_error(y_test, xgb_pred)
rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
r2 = r2_score(y_test, xgb_pred)

print(mae)
print(rmse)
print(r2)

6.684009502911685
8.897279132686942
0.47752407215041126


In [17]:
# Compare model performance in a table

results = []

models = {
    "Linear Regression": pred,
    "Decision Tree": dt_pred,
    "Random Forest": rf_pred,
    "Bagging": bagging_pred,
    "SVR": svr_pred,
    "XGBoost": xgb_pred,
}

for model_name, y_pred in models.items():
    results.append({
        "Model": model_name,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R²": r2_score(y_test, y_pred)
    })

# Voting Regressor
try:
    voting = VotingRegressor([
        ("lr", LinearRegression()),
        ("dt", DecisionTreeRegressor(random_state=42)),
        ("rf", RandomForestRegressor(n_estimators=200, random_state=42)),
        ("xgb", xgb.XGBRegressor(
            objective="reg:squarederror",
            n_estimators=200,
            learning_rate=0.1,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42
        ))
    ])
    voting.fit(X_train, y_train)
    voting_pred = voting.predict(X_test)

    results.append({
        "Model": "Voting",
        "MAE": mean_absolute_error(y_test, voting_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, voting_pred)),
        "R²": r2_score(y_test, voting_pred)
    })
except Exception as e:
    print("Voting model could not be evaluated:", e)

results_df = pd.DataFrame(results)
results_df

,Model,MAE,RMSE,R²
0,Linear Regression,9.217208,11.469379,0.131776
1,Decision Tree,6.882729,9.378589,0.419467
2,Random Forest,6.818178,9.222290,0.438656
3,Bagging,6.831091,9.222888,0.438583
4,SVR,9.916575,12.391158,-0.013388
5,XGBoost,6.684010,8.897279,0.477524
6,Voting,6.847249,9.128804,0.449979


In [18]:
Path("../data/processed").mkdir(parents=True, exist_ok=True)
results_df.to_excel(
    "../data/processed/results_data5.xlsx",
    index=False
)